---
title: "Raster data, hands-on"
subtitle: "Same Salzburg AOI, three cloud-native access patterns to Sentinel-2, plus two NDVI datacubes over the growing season"
date: today
format:
  html:
    toc: true
    code-fold: false
    code-tools: true
    code-line-numbers: true
    code-overflow: wrap
    code-copy: true
    df-print: paged
    link-external-icon: true
    link-external-newwindow: true
    smooth-scroll: true
    embed-resources: true
    html-table-processing: none
execute:
  echo: true
  warning: false
  cache: false
  freeze: false
jupyter: python3
---

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kshitijrajsharma/cng-workshop-materials/blob/master/notebooks/02_raster.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/kshitijrajsharma/cng-workshop-materials/master?filepath=notebooks/02_raster.ipynb)
[![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-F37626?logo=jupyter&logoColor=white)](https://github.com/kshitijrajsharma/cng-workshop-materials/raw/master/notebooks/02_raster.ipynb)
[![Repo](https://img.shields.io/badge/Source-GitHub-181717?logo=github&logoColor=white)](https://github.com/kshitijrajsharma/cng-workshop-materials)

**What you'll do.** Find a clean Sentinel-2 scene over Salzburg from a STAC catalog, then compute the same two indices (NDVI and NDWI) three different ways: from individual band **COGs** (lazy HTTP range reads via `rioxarray`), from a single **Zarr** product (one open, slice in array coordinates) on ESA's EOPF Sample Service, and through **VirtuGhan** which runs the formula directly against cloud-resident COGs and only returns the result. Finish with two NDVI **datacubes** over the 2025 growing season, one built by hand with xarray, one produced by VirtuGhan, and compare the seasonal trends.

In [ ]:
# | eval: false
# Uncomment and run this cell once if you are on Colab or a fresh kernel.
!pip install pystac-client rioxarray xarray "zarr>=3" virtughan matplotlib pyproj shapely pandas aiohttp fsspec  # noqa: E501

In [ ]:
%matplotlib inline
import json
import os
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import rioxarray  # noqa: F401  registers the .rio accessor
import xarray as xr
from IPython.display import Image, display
from pystac_client import Client
from shapely.geometry import box
from shapely.ops import transform as shp_transform

REPO_RAW = "https://raw.githubusercontent.com/kshitijrajsharma/cng-workshop-materials/master"
AOI_URL = f"{REPO_RAW}/data/aoi/salzburg_bbox.json"
BBOX = tuple(json.loads(urllib.request.urlopen(AOI_URL).read())["bbox"])
xmin, ymin, xmax, ymax = BBOX

TARGET_GRID = "MGRS-33UUP"
UTM_CRS = "EPSG:32633"
to_utm = pyproj.Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True).transform
xmin_u, ymin_u, xmax_u, ymax_u = shp_transform(to_utm, box(*BBOX)).bounds

OUT = Path("../data/outputs") if Path.cwd().name == "notebooks" else Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)
print(f"AOI bbox (WGS84)   = {BBOX}")
print(f"AOI bbox (UTM 33N) = ({xmin_u:.0f}, {ymin_u:.0f}, {xmax_u:.0f}, {ymax_u:.0f})")
print(f"target MGRS tile   = {TARGET_GRID}")

::: {.callout-note}
**Why pin to one MGRS tile?** Sentinel-2 is published as 100 km MGRS tiles on a fixed UTM grid. Salzburg sits where four tiles overlap (`33UUP, 33TUN, 32TQT, 32UQU`). Pinning to `MGRS-33UUP` means every scene the catalog returns shares one CRS and one pixel grid, so `xr.concat` later can stack them into a cube without reprojection. We come back to this trade-off in §5.
:::

In [ ]:
def plot_indices(ndvi, ndwi, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    ndvi.plot(ax=axes[0], cmap="RdYlGn", vmin=-0.2, vmax=0.8, cbar_kwargs={"label": "NDVI"})
    axes[0].set_title(f"NDVI, {title}")
    axes[0].set_aspect("equal")
    ndwi.plot(ax=axes[1], cmap="RdBu", vmin=-0.5, vmax=0.5, cbar_kwargs={"label": "NDWI"})
    axes[1].set_title(f"NDWI, {title}")
    axes[1].set_aspect("equal")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## 1. Pick a clean Sentinel-2 scene over Salzburg (STAC)

Earth Search by Element84 is a public STAC API in front of Sentinel-2 L2A on AWS Open Data. We filter by bbox, date window, cloud cover, and the MGRS tile that fully covers our AOI, then sort by cloud cover and take the cleanest item. The COG section in §2 uses this exact scene; the Zarr section in §3 runs its own STAC query against the EOPF catalog because that catalog is a sparser sample; the VirtuGhan section in §4 uses a small ±3-day window around the same date. They are not byte-identical scenes, the point is to compare access patterns.

In [ ]:
ES = Client.open("https://earth-search.aws.element84.com/v1")
search = ES.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2025-06-01/2025-09-30",
    query={"eo:cloud_cover": {"lt": 10}, "grid:code": {"eq": TARGET_GRID}},
    limit=20,
)
items = sorted(search.items(), key=lambda i: i.properties["eo:cloud_cover"])
item = items[0]
DATE = item.datetime.date().isoformat()
print(f"items returned = {len(items)}")
print(f"picked id      = {item.id}")
print(f"date           = {DATE}, cloud = {item.properties['eo:cloud_cover']:.2f}%")

## 2. COG access, fetch only the bands and pixels you need

Each Sentinel-2 band on Earth Search is a **Cloud Optimized GeoTIFF**. The COG layout (tile-aligned, internally indexed) lets `rioxarray.open_rasterio(...).rio.clip_box(...)` issue HTTP range requests for only the bytes covering the AOI. Three band assets, three lazy reads, only the AOI window crosses the wire.

We compute two indices from the bands. **NDVI** = (NIR − Red) / (NIR + Red), the standard vegetation greenness signal, high where chlorophyll is active. **NDWI** = (Green − NIR) / (Green + NIR), highlights open water and saturated surfaces. Both are unitless ratios in [-1, 1].

In [ ]:
def cog_band(stac_item, asset_key):
    da = rioxarray.open_rasterio(stac_item.assets[asset_key].href, masked=True).squeeze(drop=True)
    return da.rio.clip_box(*BBOX, crs="EPSG:4326").astype("float32")


t0 = time.perf_counter()
red_cog = cog_band(item, "red")
nir_cog = cog_band(item, "nir")
green_cog = cog_band(item, "green")
ndvi_cog = (nir_cog - red_cog) / (nir_cog + red_cog)
ndwi_cog = (green_cog - nir_cog) / (green_cog + nir_cog)
t_cog = time.perf_counter() - t0
print(f"COG path: {t_cog:.2f} s, AOI shape = {red_cog.shape}")

In [ ]:
plot_indices(ndvi_cog, ndwi_cog, f"COG, Earth Search, {DATE}")

## 3. Zarr access, the EOPF Sample Service

ESA's **EOPF Sample Service** republishes Sentinel-2 L2A as **Zarr** stores, indexed by a STAC API at [stac.core.eopf.eodc.eu](https://stac.core.eopf.eodc.eu/). It is a sample, not the full archive, so it does not always hold the exact date Earth Search picked. We re-run STAC on the EOPF catalog over the same window and grid, and take the cleanest scene it does publish. All 10 m bands live in one Zarr group, so a single `xr.open_dataset(..., group="/measurements/reflectance/r10m")` gives us `b02, b03, b04, b08` together. Slicing in array coordinates fetches only the chunks that intersect the AOI.

In [ ]:
EOPF = Client.open("https://stac.core.eopf.eodc.eu/")
eopf_search = EOPF.search(
    collections="sentinel-2-l2a",
    bbox=BBOX,
    datetime="2025-05-01T00:00:00Z/2025-09-30T23:59:59Z",
    query={"eo:cloud_cover": {"lt": 20}, "grid:code": {"eq": TARGET_GRID}},
    limit=20,
)
eopf_items = sorted(eopf_search.items(), key=lambda i: i.properties.get("eo:cloud_cover", 100))
eopf_item = eopf_items[0]
EOPF_DATE = eopf_item.datetime.date().isoformat()
product_url = eopf_item.assets["product"].href
print(f"EOPF items in window = {len(eopf_items)}")
print(f"picked id            = {eopf_item.id}")
print(f"date                 = {EOPF_DATE}, cloud = {eopf_item.properties['eo:cloud_cover']:.2f}%")
print(f"Zarr product         = {product_url}")

In [ ]:
t0 = time.perf_counter()
r10m = xr.open_dataset(
    product_url,
    engine="zarr",
    group="/measurements/reflectance/r10m",
    consolidated=True,
)
aoi = r10m.sel(x=slice(xmin_u, xmax_u), y=slice(ymax_u, ymin_u)).load()
ndvi_zarr = (aoi.b08 - aoi.b04) / (aoi.b08 + aoi.b04)
ndwi_zarr = (aoi.b03 - aoi.b08) / (aoi.b03 + aoi.b08)
t_zarr = time.perf_counter() - t0
print(f"Zarr path: {t_zarr:.2f} s, AOI shape = {aoi.b04.shape}")
print(f"COG path from §2 for reference: {t_cog:.2f} s (different scene, same AOI)")

In [ ]:
plot_indices(ndvi_zarr, ndwi_zarr, f"Zarr, EOPF, {EOPF_DATE}")

## 4. VirtuGhan, on-the-fly index from cloud-resident COGs

[VirtuGhan](https://github.com/virtughan/virtughan) takes a STAC catalog, a bbox, and a band-math formula, then computes the index directly from cloud-resident COGs and returns just the result. The per-band fetching, masking, and aggregation all run server-side. We give it the same AOI and a tight one-week window around our COG date, ask for the median-aggregated composite, and run it twice: once for NDVI, once for NDWI.

In [ ]:
from virtughan.engine import VirtughanProcessor

VG_START = (pd.Timestamp(DATE) - pd.Timedelta(days=3)).date().isoformat()
VG_END = (pd.Timestamp(DATE) + pd.Timedelta(days=3)).date().isoformat()


def virtughan_composite(band1, band2, label):
    out_dir = OUT / f"vg_single_{label}"
    out_dir.mkdir(parents=True, exist_ok=True)
    proc = VirtughanProcessor(
        bbox=list(BBOX),
        start_date=VG_START,
        end_date=VG_END,
        cloud_cover=30,
        formula="(band1 - band2) / (band1 + band2)",
        band1=band1,
        band2=band2,
        operation="median",
        timeseries=False,
        output_dir=str(out_dir),
        smart_filter=False,
    )
    proc.compute()
    return out_dir


t0 = time.perf_counter()
ndvi_dir = virtughan_composite("nir", "red", "ndvi")
ndwi_dir = virtughan_composite("green", "nir", "ndwi")
print(f"VirtuGhan composite ({VG_START} to {VG_END}): {time.perf_counter() - t0:.2f} s")

In [ ]:
ndvi_vg = rioxarray.open_rasterio(
    ndvi_dir / "custom_band_output_aggregate.tif",
    masked=True,
).squeeze(drop=True)
ndwi_vg = rioxarray.open_rasterio(
    ndwi_dir / "custom_band_output_aggregate.tif",
    masked=True,
).squeeze(drop=True)
plot_indices(ndvi_vg, ndwi_vg, f"VirtuGhan median, {VG_START} to {VG_END}")

## 5. Datacube, NDVI time series with xarray + COG

A **datacube** is just a multi-dimensional array with named axes. Until now we worked with a `(y, x)` slice for a single date. Stack N dates along a new `time` axis and you get a `(time, y, x)` cube. Take the spatial median per date and you collapse `(y, x)` into a single number, yielding a `(time,)` series, the seasonal trend.

Below: search L2A for the Salzburg growing season, build the `(time, y, x)` cube by lazy-opening each scene's COG bands, take the spatial median per date, fit a linear trend, and plot in the same style as the VirtuGhan chart further down so the two are directly comparable.

In [ ]:
TS_START, TS_END = "2025-05-01", "2025-09-30"
ts_search = ES.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=f"{TS_START}/{TS_END}",
    query={"eo:cloud_cover": {"lt": 20}, "grid:code": {"eq": TARGET_GRID}},
    limit=50,
)
ts_items = sorted(ts_search.items(), key=lambda i: i.datetime)
print(f"timeseries scenes (cloud < 20%, {TS_START} to {TS_END}) = {len(ts_items)}")

In [ ]:
def ndvi_for_item(it):
    red = cog_band(it, "red")
    nir = cog_band(it, "nir")
    ndvi = (nir - red) / (nir + red)
    return ndvi.expand_dims(time=[pd.Timestamp(it.datetime)])


ndvi_cube = xr.concat([ndvi_for_item(it) for it in ts_items], dim="time")
print(f"NDVI cube shape = {ndvi_cube.shape} (time, y, x)")

In [ ]:
ndvi_median_xr = ndvi_cube.median(["x", "y"], skipna=True)
dates = pd.to_datetime(ndvi_median_xr.time.values)
values = ndvi_median_xr.values
dates_numeric = np.arange(len(dates))
slope, intercept = np.polyfit(dates_numeric, values, 1)
trend_line = slope * dates_numeric + intercept

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dates, values, marker="o", color="#1f77b4", label="Median Value")
ax.plot(dates, trend_line, "--", color="red", label="Trend Line")
ax.set_title("Median Value Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Median Value")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y%m%d"))
ax.tick_params(axis="x", rotation=45)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
fig.tight_layout()
display(fig)
plt.close(fig)

::: {.callout-note}
This cube uses only `MGRS-33UUP` scenes so every entry shares one UTM grid and `xr.concat` stitches cleanly. Salzburg is also covered by **33TUN, 32TQT, and 32UQU**, each contributing a few extra acquisition dates per season. The VirtuGhan chart in the next section pulls from all four tiles (reprojecting internally) and ends up with 14 dates vs our 8. Reaching those extra dates from xarray means per-scene reprojection plus filtering out edge-of-tile scenes where the AOI overlap is a thin sliver, which is exactly the kind of plumbing VirtuGhan does for you.
:::

## 6. Datacube, NDVI time series with VirtuGhan

Same date window, same AOI, same Sentinel-2 source, but the per-date NDVI is computed *inside VirtuGhan* against the STAC catalog. We just consume the per-date GeoTIFFs it writes and replicate the same trend chart.

In [ ]:
vg_ts_dir = OUT / "vg_ts_ndvi"
vg_ts_dir.mkdir(parents=True, exist_ok=True)

t0 = time.perf_counter()
proc = VirtughanProcessor(
    bbox=list(BBOX),
    start_date=TS_START,
    end_date=TS_END,
    cloud_cover=20,
    formula="(band1 - band2) / (band1 + band2)",
    band1="nir",
    band2="red",
    operation="median",
    timeseries=True,
    output_dir=str(vg_ts_dir),
    smart_filter=False,
    workers=os.cpu_count() or 4,
)
proc.compute()
print(f"VirtuGhan timeseries: {time.perf_counter() - t0:.2f} s")

VirtuGhan also writes `values_over_time.png` next to the per-date GeoTIFFs, with the spatial-median NDVI and a fitted trend line. Display it directly.

In [ ]:
Image(filename=str(vg_ts_dir / "values_over_time.png"))

## References

- Earth Search v1: <https://element84.com/geospatial/introducing-earth-search-v1-new-datasets-now-available/>
- pystac-client: <https://pystac-client.readthedocs.io/>
- Cloud Optimized GeoTIFF: <https://cogeo.org/>
- EOPF Sample Service: <https://eopf-toolkit.github.io/eopf-101/>
- VirtuGhan: <https://github.com/virtughan/virtughan>
- Sentinel-2 L2A bands: <https://sentiwiki.copernicus.eu/web/s2-products>